# Project 1 - The Tool Loop by Hand

#### Objective: Implement the tool-calling loop yourself, without create_agent, so you understand exactly what an agent does on every turn.

##### Test
```How much is 250 GBP in JPY? Then convert that JPY amount into USD.```

##### Expected Output
```
Tool call 1: convert_currency({'amount': 250, 'from_ccy': 'GBP', 'to_ccy': 'JPY'})
Tool call 2: convert_currency({'amount': 47500.0, 'from_ccy': 'JPY', 'to_ccy': 'USD'})
Final: 250 GBP is 47,500 JPY, which is about 317 USD.
Total tokens: 1,284
```

#### Step 1: Prerequisties - Import Library, Load Environment

In [1]:
from dotenv import load_dotenv
import pydantic
from langchain.tools import tool
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.messages import HumanMessage, AIMessage, ToolMessage

# Load Environment Variables
load_dotenv(override=True)

True

#### Step 2: Declare Convert Currency Tool

In [12]:
# Manually Defined Currency Rates 
rates = {"USD": 1.0, "GBP": 0.7476, "EUR": 0.8916, "JPY": 157.32}

# Tool
@tool
def convert_currency(amount: float, from_ccy: str, to_ccy: str) -> str:
    """Convert the currency amount from the currency type stated in from_ccy
    to the currency type stated in to_ccy. Only currency types of 
    GBP, USD, EUR, JPY are supported. Returns the exchanged amount in str format such as '3.36GBP'.
    If any of the from or to currency type is not allowed, return 
    'Error: unsupported currency 'XYZ'. Supported: GBP, USD, EUR, JPY'"""
    # Return Unsupported Currency Type
    if from_ccy not in rates:
        return f"Error: unsupported currency '{from_ccy}'. Supported: GBP, USD, EUR, JPY"
    if to_ccy not in rates:
        return f"Error: unsupported currency '{to_ccy}'. Supported: GBP, USD, EUR, JPY"
    exchanged_amt = (amount/rates[from_ccy]) * rates[to_ccy]
    return f"{exchanged_amt:.2f}{to_ccy}"

#### Step 3: Create Agent Using Convert Currency Tool

In [13]:
model = init_chat_model("openai:gpt-4o-mini")
model_with_tools = model.bind_tools([convert_currency])

#### Step 4: Call The Model

In [18]:
# Human Messages
messages = [HumanMessage(content="Convert 100 USD to MMK. Then convert it to THB")]

# Response
response = model_with_tools.invoke(messages)
messages.append(response)

# Tool Call - Max 5 Calls, 
count = 1
while response.tool_calls and count < 5:
    for call in response.tool_calls:
        if call["name"] == "convert_currency":
            result = convert_currency.invoke(call["args"])
        else:
            result = f"Error: unknown tool '{call['name']}'"
        messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
    response = model_with_tools.invoke(messages)
    messages.append(response)
    count += 1
    print("Total Tokens For Call: ", response.usage_metadata["total_tokens"])

print(response.content)

Total Tokens For Call:  311
I can only convert currencies among GBP, USD, EUR, and JPY. Please let me know if you'd like to convert USD to one of these supported currencies instead.


In [19]:
for m in messages:
    m.pretty_print()

================================ Human Message =================================

Convert 100 USD to MMK. Then convert it to THB
================================== Ai Message ==================================
Tool Calls:
  convert_currency (call_zy2uT6RmCCR9YLqxbKbCBVKU)
 Call ID: call_zy2uT6RmCCR9YLqxbKbCBVKU
  Args:
    amount: 100
    from_ccy: USD
    to_ccy: MMK
  convert_currency (call_KPuIUQl9yjH2f8MZBUnsIOoo)
 Call ID: call_KPuIUQl9yjH2f8MZBUnsIOoo
  Args:
    amount: 100
    from_ccy: USD
    to_ccy: THB
================================= Tool Message =================================

Error: unsupported currency 'MMK'. Supported: GBP, USD, EUR, JPY
================================= Tool Message =================================

Error: unsupported currency 'THB'. Supported: GBP, USD, EUR, JPY
================================== Ai Message ==================================

I can only convert currencies among GBP, USD, EUR, and JPY. Please let me know if you'd like to convert 